# 17.6 從 8-bit 降到 4-bit 會發生什麼？

# 第 17 章：量化，用更少位元表示模型

前置：第2章Linear與數值。像把連續刻度尺改成較粗的格子：先看誤差，再看真正的儲存bytes，最後才談硬體加速。quantization.py的參考Linear會反量化，不會假裝是低位元kernel。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：兩個小格裝一個byte：低四位與高四位**

本例signed加8：−3→0101，+4→1100，合成11000101。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/int4.svg")))

**先想一想：**4bit比8bit誤差一定每一格都更大嗎？

4bit只有16種編碼，8bit有256種。固定同一組weight才能看位元數取捨；理論payload還沒包括scale與packing padding。

**把直覺寫成數學：**理論payload=參數數×bits/8；對稱量化可能保留一個未用編碼。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.quantization import quantize_symmetric

w = torch.linspace(-1, 1, 33)
for bits in (8, 4):
    q, s = quantize_symmetric(w, bits)
    print(bits, "平均誤差", (w - q * s).abs().mean().item(), "理論bytes", w.numel() * bits / 8)

**如何讀結果：**平均趨勢不是每個數字的嚴格排序；比較實際任務還要轉換整模型。

**只改一件事：**只使用恰好落在4bit刻度的原數。
